# Gaussian Processes

## Overview

A Gaussian Process (GP) is a distribution over functions. It provides a flexible non-parametric regression approach that produces full uncertainty estimates (posterior predictive distributions) at every point in the input space.

**Key idea:** instead of specifying a parametric function form, a GP defines a prior over functions by specifying how similar function values should be at similar inputs — encoded in the **kernel (covariance function)**.

**Common kernels:**

| Kernel | Produces | Use when |
|---|---|---|
| RBF (squared exponential) | Infinitely smooth functions | Smooth spatial/temporal trends |
| Matérn 3/2 or 5/2 | Moderately smooth | Physical processes |
| Periodic | Repeating patterns | Seasonal data |
| Linear | Linear functions | When parametric trend suffices |

**Two hyperparameters per kernel:** length scale (how quickly correlation decays) and signal variance (amplitude of variation).

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (RBF, Matern, WhiteKernel,
    ConstantKernel, ExpSineSquared, RationalQuadratic)
from scipy import stats

rng = np.random.default_rng(42)
# Sparse monitoring data: nitrate measured at 20 sites along an elevation gradient
n_obs = 20
elev_obs = np.sort(rng.uniform(50, 400, n_obs))
true_fn  = lambda x: 5 + 0.01*(x-200) - 0.00005*(x-200)**2
nitrate_obs = true_fn(elev_obs) + rng.normal(0, 0.4, n_obs)
X_obs = elev_obs.reshape(-1,1)
y_obs = nitrate_obs
# Prediction grid
X_pred = np.linspace(50, 400, 200).reshape(-1,1)
print(f"Observed: {n_obs} sites, elevation {elev_obs.min():.0f}–{elev_obs.max():.0f}m")

---
## Fitting a GP with RBF Kernel

In [ ]:
# RBF + WhiteKernel (noise)
kernel = ConstantKernel(1.0) * RBF(length_scale=80.0) + WhiteKernel(noise_level=0.2)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10,
                               normalize_y=True, random_state=42)
gp.fit(X_obs, y_obs)
print(f"Optimised kernel: {gp.kernel_}")
print(f"Log marginal likelihood: {gp.log_marginal_likelihood_value_:.3f}")
y_mean, y_std = gp.predict(X_pred, return_std=True)
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(X_obs, y_obs, color="#e74c3c", s=50, zorder=5, label="Observations")
ax.plot(X_pred, true_fn(X_pred), color="navy", lw=1.5, linestyle="--", label="True function")
ax.plot(X_pred, y_mean, color="steelblue", lw=2, label="GP mean")
ax.fill_between(X_pred.ravel(), y_mean-2*y_std, y_mean+2*y_std,
                alpha=0.3, color="steelblue", label="95% credible band")
ax.set_xlabel("Elevation (m)"); ax.set_ylabel("Nitrate (mg/L)")
ax.set_title("Gaussian Process Regression: Uncertainty Widens in Data-Sparse Regions")
ax.legend(); plt.tight_layout(); plt.show()

---
## Kernel Comparison

In [ ]:
kernels = {
    "RBF":        ConstantKernel(1.0)*RBF(80) + WhiteKernel(0.2),
    "Matern 3/2": ConstantKernel(1.0)*Matern(length_scale=80, nu=1.5) + WhiteKernel(0.2),
    "Matern 5/2": ConstantKernel(1.0)*Matern(length_scale=80, nu=2.5) + WhiteKernel(0.2),
    "Rat. Quad.": ConstantKernel(1.0)*RationalQuadratic(length_scale=80) + WhiteKernel(0.2),
}
fig, axes = plt.subplots(2,2,figsize=(13,8))
for ax, (name, ker) in zip(axes.flatten(), kernels.items()):
    gp_k = GaussianProcessRegressor(kernel=ker, n_restarts_optimizer=5,
                                     normalize_y=True, random_state=42).fit(X_obs, y_obs)
    ym, ys = gp_k.predict(X_pred, return_std=True)
    ax.scatter(X_obs, y_obs, color="#e74c3c", s=30, zorder=5)
    ax.plot(X_pred, true_fn(X_pred), color="navy", lw=1, linestyle="--", alpha=0.6)
    ax.plot(X_pred, ym, color="steelblue", lw=2)
    ax.fill_between(X_pred.ravel(), ym-2*ys, ym+2*ys, alpha=0.3, color="steelblue")
    ax.set_title(f"{name}\nLML={gp_k.log_marginal_likelihood_value_:.1f}")
    ax.set_xlabel("Elevation"); ax.set_ylabel("Nitrate")
plt.suptitle("Kernel Comparison: Shape of Uncertainty Changes")
plt.tight_layout(); plt.show()

---
## GP for Spatial Interpolation

In [ ]:
# 2D example: predict nitrate at unsampled locations
rng2 = np.random.default_rng(0)
n_sp = 40
easting  = rng2.uniform(0, 100, n_sp)
northing = rng2.uniform(0, 100, n_sp)
z_true   = np.sin(easting/20) + np.cos(northing/15)
z_obs    = z_true + rng2.normal(0, 0.2, n_sp)
X_sp = np.column_stack([easting, northing])
ker_2d = ConstantKernel(1.0)*RBF(length_scale=[20.0,20.0]) + WhiteKernel(0.1)
gp_2d = GaussianProcessRegressor(kernel=ker_2d, n_restarts_optimizer=5,
                                  normalize_y=True, random_state=42)
gp_2d.fit(X_sp, z_obs)
# Prediction grid
eg, ng = np.meshgrid(np.linspace(0,100,50), np.linspace(0,100,50))
X_grid = np.column_stack([eg.ravel(), ng.ravel()])
z_mean, z_std = gp_2d.predict(X_grid, return_std=True)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, vals, title in zip(axes,[z_mean,z_std],["GP Mean","GP Uncertainty (SD)"]):
    c = ax.contourf(eg, ng, vals.reshape(50,50), levels=20, cmap="viridis")
    ax.scatter(easting, northing, c="white", s=20, zorder=5, edgecolors="black", lw=0.5)
    plt.colorbar(c, ax=ax); ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
# Posterior predictive samples (draw function realisations from the GP)
samples = gp.sample_y(X_pred, n_samples=5, random_state=42)
fig, ax = plt.subplots(figsize=(10,5))
for s in samples.T:
    ax.plot(X_pred, s, lw=1, alpha=0.5, color="steelblue")
ax.scatter(X_obs, y_obs, color="#e74c3c", s=50, zorder=5, label="Observations")
ax.plot(X_pred, true_fn(X_pred), color="navy", lw=2, linestyle="--", label="True function")
ax.set_xlabel("Elevation (m)"); ax.set_ylabel("Nitrate (mg/L)")
ax.set_title("GP Posterior Samples: 5 Plausible Functions Given the Data")
ax.legend(); plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Not optimising kernel hyperparameters**  
The default length scale may be far from optimal for the data at hand. Always set `n_restarts_optimizer >= 5` to avoid local optima in marginal likelihood optimisation, and inspect the optimised kernel parameters before trusting predictions.

**2. Using RBF kernel when the process has non-smooth features**  
The RBF kernel produces infinitely differentiable (unrealistically smooth) functions. Most physical and ecological processes are better described by Matérn 3/2 or 5/2 kernels, which allow for less smooth behaviour.

**3. Treating GP uncertainty bands as frequentist confidence intervals**  
GP uncertainty bands are Bayesian posterior credible intervals, conditional on the kernel hyperparameters. They do not account for uncertainty in the hyperparameters themselves (which can be large with sparse data). Full Bayesian GPs marginalise over hyperparameters.

**4. Applying GP regression to large datasets without approximations**  
Exact GP regression requires inverting an n×n kernel matrix — O(n³) time and O(n²) memory. For n > 5,000 this becomes infeasible. Use sparse GPs (`sklearn.gaussian_process` with inducing points), approximate kernels, or GPyTorch for large data.

**5. Extrapolating GP predictions far beyond the training data range**  
In data-sparse regions, the GP reverts to the prior mean with the prior variance. Extrapolations beyond the observed range should be accompanied by explicitly widening uncertainty bands and a statement that predictions are prior-dominated.
---
*python_methods_library - Samantha McGarrigle*